# 03 - Clustering Sweep

Runs the 12-configuration reducer × clusterer matrix on a manageable sample and inspects internal validity metrics.

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import pandas as pd
import plotly.express as px
from src import features, ingest, multi_config, preprocess

PRESENTATION = ("AAA", "2014J")
SAMPLE_N = 300

In [ ]:
tables = ingest.run(*PRESENTATION)
X = features.run(tables)
if len(X) > SAMPLE_N:
    X = X.sample(SAMPLE_N, random_state=42).sort_values("id_student").reset_index(drop=True)
ids, X_scaled, *_rest, columns, clean_features = preprocess.preprocess(X)
print(X_scaled.shape)

In [ ]:
metrics_df, labels_by_config, reductions = multi_config.run_all(X_scaled)
display_cols = [
    "config_id", "reducer", "clusterer", "k", "k_effective", "noise_ratio",
    "silhouette", "davies_bouldin", "calinski_harabasz", "dbcv"
]
metrics_df[display_cols].sort_values(["clusterer", "reducer"])

In [ ]:
primary = metrics_df[metrics_df["clusterer"].ne("hdbscan")].copy()
fig = px.scatter(
    primary,
    x="silhouette",
    y="calinski_harabasz",
    color="clusterer",
    symbol="reducer",
    size="k",
    hover_name="config_id",
    title="Internal Validity Across Primary Configurations",
)
fig.show()

In [ ]:
hdbscan_rows = metrics_df[metrics_df["clusterer"].eq("hdbscan")]
hdbscan_rows[["config_id", "reducer", "k_effective", "noise_ratio", "dbcv", "hdbscan_impl"]]

HDBSCAN is retained as a density-based comparison track. If it produces high noise on a presentation, discuss it as a data-density mismatch rather than forcing it into the primary selector.